#########################################################################################################
Instalacion de librerias
#########################################################################################################

In [48]:
%%capture
!pip install setuptools==68.0.0 ydata-profiling scikit-learn pandas numpy missingno



In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msn

from ydata_profiling import ProfileReport
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

pd.set_option("display.max_columns",25)
pd.set_option("display.max_rows",60)
sns.set_style("whitegrid")
print("Libraries imported successfully!")

Libraries imported successfully!


#########################################################################################################
PERFILADO DE DATOS
#########################################################################################################

In [50]:
df = pd.read_csv("datos.csv", on_bad_lines='skip')
print("shape of the dataset:", df.shape)
print("columns in the dataset:", df.columns)
print("shape of the dataset:", df.shape)

shape of the dataset: (1055, 7)
columns in the dataset: Index([''Flight phase'', ''Flight type'', ''Crash site'', ''Crew on board'',
       ''Pax on board'', ''Total fatalities'', ''Crash cause''],
      dtype='object')
shape of the dataset: (1055, 7)


In [51]:
%%capture
profile = ProfileReport(df, title="Reporte de Perfilado de Datos", explorative=True)


profile.to_file("reporte.html")
print

In [52]:
missing_summary = pd.DataFrame({
    "Nulos": df.isnull().sum(),
    "Porcentaje (%)": (df.isnull().mean() * 100).round(3)
})

duplicate_count = df.duplicated().sum()
duplicate_pct = duplicate_count / len(df) * 100
unique_counts = df.nunique(dropna=False)

print("Diagnóstico general de calidad del dataset")
print("-" * 70)
print(f"Filas:", df.shape[0])
print(f"Columnas:", df.shape[1])
print(f"Celdas totales: {df.size}")
print(f"Celdas con nulos: {df.isnull().sum().sum()} ({(df.isnull().mean().mean() * 100):.3f}%)")
print(f"Filas duplicadas: {duplicate_count} ({duplicate_pct:.3f}%)")
print()

Diagnóstico general de calidad del dataset
----------------------------------------------------------------------
Filas: 1055
Columnas: 7
Celdas totales: 7385
Celdas con nulos: 3 (0.041%)
Filas duplicadas: 123 (11.659%)



In [53]:

print("Tipos de datos:")
print(df.dtypes)
print()
print("Valores únicos por columna:")
print(unique_counts)
print()


Tipos de datos:
'Flight phase'         object
'Flight type'          object
'Crash site'           object
'Crew on board'       float64
'Pax on board'        float64
'Total fatalities'     object
'Crash cause'          object
dtype: object

Valores únicos por columna:
'Flight phase'          6
'Flight type'          23
'Crash site'            4
'Crew on board'       302
'Pax on board'        414
'Total fatalities'      4
'Crash cause'           5
dtype: int64



In [54]:
print("Resumen de nulos por columna:")
print(missing_summary)
print()
print("Top 3 valores por columna categórica:")
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(3))

Resumen de nulos por columna:
                    Nulos  Porcentaje (%)
'Flight phase'          3           0.284
'Flight type'           0           0.000
'Crash site'            0           0.000
'Crew on board'         0           0.000
'Pax on board'          0           0.000
'Total fatalities'      0           0.000
'Crash cause'           0           0.000

Top 3 valores por columna categórica:

'Flight phase':
'Flight phase'
'Landing (descent or approach)'    569
'Takeoff (climb)'                  264
Flight                             205
Name: count, dtype: int64

'Flight type':
'Flight type'
'Scheduled Revenue Flight'    564
Cargo                         112
Military                       96
Name: count, dtype: int64

'Crash site':
'Crash site'
'Airport (less than 10 km from airport)'    733
Mountains                                   279
City                                         40
Name: count, dtype: int64

'Total fatalities':
'Total fatalities'
11-40     397
0-10      

In [55]:
rows, cols = df.shape
total_cells = rows * cols
null_cells = df.isnull().sum().sum()
null_pct = null_cells / total_cells * 100
completitud_total = 100 - null_pct

duplicate_count = df.duplicated().sum()
unique_rows = rows - duplicate_count
unique_rows_pct = unique_rows / rows * 100

type_counts = df.apply(lambda col: col.map(type).nunique())
mixed_columns = type_counts[type_counts > 1].index.tolist()
consistency_pct = 100 - len(mixed_columns) / cols * 100

valid_counts = {}
for col in df.columns:
    ser = df[col]
    non_null = ser.dropna()
    if non_null.empty:
        valid_counts[col] = 0
        continue

    if np.issubdtype(ser.dtype, np.number):
        valid_counts[col] = pd.to_numeric(non_null, errors="coerce").notna().sum()
    elif np.issubdtype(ser.dtype, np.datetime64):
        valid_counts[col] = non_null.shape[0]
    else:
        num = pd.to_numeric(non_null, errors="coerce")
        dt = pd.to_datetime(non_null, errors="coerce", infer_datetime_format=True)
        if num.notna().mean() >= 0.9:
            valid_counts[col] = num.notna().sum()
        elif dt.notna().mean() >= 0.9:
            valid_counts[col] = dt.notna().sum()
        else:
            valid_counts[col] = non_null.shape[0]

valid_cells = sum(valid_counts.values())
validez_total = valid_cells / total_cells * 100

precision_rows = []
for col in df.select_dtypes(include="number").columns:
    values = df[col].dropna().astype(str)
    decimals = values.str.extract(r"\.(\d+)")[0].dropna().str.len()
    precision_rows.append({
        "Columna": col,
        "Mediana de decimales": int(decimals.median()) if len(decimals) else 0,
        "Valores numéricos": len(values)
    })
precision_report = pd.DataFrame(precision_rows)

date_summary = []
for col in df.columns:
    if np.issubdtype(df[col].dtype, np.datetime64):
        parsed = df[col]
    else:
        parsed = pd.to_datetime(df[col], errors="coerce", infer_datetime_format=True)
        if parsed.notna().mean() < 0.8:
            continue

    valid_dates = parsed.dropna()
    if not valid_dates.empty:
        date_summary.append({
            "Columna": col,
            "Registros válidos (%)": valid_dates.size / rows * 100,
            "Fecha mínima": valid_dates.min(),
            "Fecha máxima": valid_dates.max()
        })
date_report = pd.DataFrame(date_summary)

print("REPORTE DE CALIDAD DEL DATASET")
print("=" * 40)
print("1. Completitud:")
print(f"   - Filas: {rows}")
print(f"   - Columnas: {cols}")
print(f"   - Celdas totales: {total_cells}")
print(f"   - Celdas con nulos: {null_cells} ({null_pct:.2f}%)")
print(f"   - Completitud general: {completitud_total:.2f}%")
print()



REPORTE DE CALIDAD DEL DATASET
1. Completitud:
   - Filas: 1055
   - Columnas: 7
   - Celdas totales: 7385
   - Celdas con nulos: 3 (0.04%)
   - Completitud general: 99.96%



In [58]:
print("2. Unicidad:")
print(f"   - Filas únicas: {unique_rows} ({unique_rows_pct:.2f}%)")
print(f"   - Filas duplicadas: {duplicate_count} ({duplicate_count / rows * 100:.2f}%)")
print(f"   - Columnas con valores únicos / totales:")
print(df.nunique(dropna=False).to_string())
print()

print("3. Validez:")
print(f"   - Porcentaje de celdas válidas estimado: {validez_total:.2f}%")
print("   - Validez por columna (% de valores compatibles con tipo esperado o no nulos):")
validity_report = pd.DataFrame({
    "Columna": df.columns,
    "Porcentaje válido (%)": [valid_counts[col] / rows * 100 for col in df.columns],
    "Nulos": df[col].isnull().sum()
})
print(validity_report.set_index("Columna").to_string())
print()

2. Unicidad:
   - Filas únicas: 932 (88.34%)
   - Filas duplicadas: 123 (11.66%)
   - Columnas con valores únicos / totales:
'Flight phase'          6
'Flight type'          23
'Crash site'            4
'Crew on board'       302
'Pax on board'        414
'Total fatalities'      4
'Crash cause'           5

3. Validez:
   - Porcentaje de celdas válidas estimado: 99.96%
   - Validez por columna (% de valores compatibles con tipo esperado o no nulos):
                    Porcentaje válido (%)  Nulos
Columna                                         
'Flight phase'                   99.71564      0
'Flight type'                   100.00000      0
'Crash site'                    100.00000      0
'Crew on board'                 100.00000      0
'Pax on board'                  100.00000      0
'Total fatalities'              100.00000      0
'Crash cause'                   100.00000      0



In [59]:
print("4. Consistencia:")
print(f"   - Columnas con tipos mixtos detectados: {len(mixed_columns)} de {cols}")
if mixed_columns:
    print(f"   - Columnas mixtas: {mixed_columns}")
print(f"   - Consistencia de tipo por columna: {consistency_pct:.2f}%")
print()

print("5. Precisión:")
if not precision_report.empty:
    print("   - Precisión estimada según decimales en columnas numéricas:")
    print(precision_report.to_string(index=False))
else:
    print("   - No se identificaron columnas numéricas para evaluar precisión mediante decimales.")
print("   - Nota: la precisión real requiere datos de referencia externos.")
print()


4. Consistencia:
   - Columnas con tipos mixtos detectados: 1 de 7
   - Columnas mixtas: ["'Flight phase'"]
   - Consistencia de tipo por columna: 85.71%

5. Precisión:
   - Precisión estimada según decimales en columnas numéricas:
        Columna  Mediana de decimales  Valores numéricos
'Crew on board'                     1               1055
 'Pax on board'                     1               1055
   - Nota: la precisión real requiere datos de referencia externos.



#########################################################################################################
LIMPIEZA DE DATOS
#########################################################################################################

In [62]:
print("Iniciando limpieza y mejora de datos...")

# 1. Normalizar nombres de columnas y cadenas de texto
clean_cols = df.columns.str.strip().str.strip("'\"")
df.columns = clean_cols
string_cols = df.select_dtypes(include=["string", "object"]).columns
for col in string_cols:
    df[col] = df[col].astype("string").str.strip().str.strip("'\"")

# 2. Imputar valores faltantes en variables categóricas relevantes
if df["Flight phase"].isna().any():
    fill_value = df["Flight phase"].mode(dropna=True)
    fill_value = fill_value.iloc[0] if not fill_value.empty else "Unknown"
    df["Flight phase"] = df["Flight phase"].fillna(fill_value)

# 3. Asegurar tipos numéricos para las columnas de conteo
for col in ["Crew on board", "Pax on board"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 4. Mejorar la variable de fallecimientos creando una versión numérica aproximada
fatalities_map = {
    "0-10": 5,
    "11-40": 25,
    "41-100": 70,
    "101+": 120
}
df["Fatalities approx"] = df["Total fatalities"].map(fatalities_map)

# 5. Convertir campos categóricos a tipo category para eficiencia y consistencia
for col in ["Flight phase", "Flight type", "Crash site", "Crash cause", "Total fatalities"]:
    df[col] = df[col].astype("category")

# 6. Eliminar duplicados y reindexar
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
removed_duplicates = before - after

# 7. Guardar el dataset limpio
output_path = "datos_limpios.csv"
df.to_csv(output_path, index=False)

print("Limpieza completada.")
print(f"  - Filas originales: {before}")
print(f"  - Filas después de limpiar duplicados: {after}")
print(f"  - Duplicados eliminados: {removed_duplicates}")
print(f"  - Nulos restantes por columna:\n{df.isnull().sum()}\n")
print(f"  - Dataset limpio guardado en: {output_path}")


Iniciando limpieza y mejora de datos...
Limpieza completada.
  - Filas originales: 931
  - Filas después de limpiar duplicados: 931
  - Duplicados eliminados: 0
  - Nulos restantes por columna:
Flight phase         0
Flight type          0
Crash site           0
Crew on board        0
Pax on board         0
Total fatalities     0
Crash cause          0
Fatalities approx    0
dtype: int64

  - Dataset limpio guardado en: datos_limpios.csv
